In [ ]:
!nvidia-smi

Fri Mar  7 16:06:55 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install transformers
!pip install datasets
!pip install keras
!pip install tensorflow
!pip install transformers torch
!pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import pandas as pd
import matplotlib
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['axes.unicode_minus'] = False
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Dense, Embedding, LSTM, SpatialDropout1D
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from keras.callbacks import EarlyStopping
from keras.layers import Dropout
import jieba as jb
import re

In [ ]:
import spacy
from transformers import pipeline

In [ ]:
from datasets import load_dataset

ds = load_dataset("legacy-datasets/banking77")
train_data = ds["train"]
dev_data = ds["train"]
test_data = ds["test"]
# Check first entry
print(ds)
print(ds["train"][0])
print(ds["test"][0])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/14.4k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/298k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/93.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3080 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})
{'text': 'I am still waiting on my card?', 'label': 11}
{'text': 'How do I locate my card?', 'label': 11}


In [ ]:
category_mapping_num_to_category = {
    0: "activate_my_card",
    1: "age_limit",
    2: "apple_pay_or_google_pay",
    3: "atm_support",
    4: "automatic_top_up",
    5: "balance_not_updated_after_bank_transfer",
    6: "balance_not_updated_after_cheque_or_cash_deposit",
    7: "beneficiary_not_allowed",
    8: "cancel_transfer",
    9: "card_about_to_expire",
    10: "card_acceptance",
    11: "card_arrival",
    12: "card_delivery_estimate",
    13: "card_linking",
    14: "card_not_working",
    15: "card_payment_fee_charged",
    16: "card_payment_not_recognised",
    17: "card_payment_wrong_exchange_rate",
    18: "card_swallowed",
    19: "cash_withdrawal_charge",
    20: "cash_withdrawal_not_recognised",
    21: "change_pin",
    22: "compromised_card",
    23: "contactless_not_working",
    24: "country_support",
    25: "declined_card_payment",
    26: "declined_cash_withdrawal",
    27: "declined_transfer",
    28: "direct_debit_payment_not_recognised",
    29: "disposable_card_limits",
    30: "edit_personal_details",
    31: "exchange_charge",
    32: "exchange_rate",
    33: "exchange_via_app",
    34: "extra_charge_on_statement",
    35: "failed_transfer",
    36: "fiat_currency_support",
    37: "get_disposable_virtual_card",
    38: "get_physical_card",
    39: "getting_spare_card",
    40: "getting_virtual_card",
    41: "lost_or_stolen_card",
    42: "lost_or_stolen_phone",
    43: "order_physical_card",
    44: "passcode_forgotten",
    45: "pending_card_payment",
    46: "pending_cash_withdrawal",
    47: "pending_top_up",
    48: "pending_transfer",
    49: "pin_blocked",
    50: "receiving_money",
    51: "Refund_not_showing_up",
    52: "request_refund",
    53: "reverted_card_payment?",
    54: "supported_cards_and_currencies",
    55: "terminate_account",
    56: "top_up_by_bank_transfer_charge",
    57: "top_up_by_card_charge",
    58: "top_up_by_cash_or_cheque",
    59: "top_up_failed",
    60: "top_up_limits",
    61: "top_up_reverted",
    62: "topping_up_by_card",
    63: "transaction_charged_twice",
    64: "transfer_fee_charged",
    65: "transfer_into_account",
    66: "transfer_not_received_by_recipient",
    67: "transfer_timing",
    68: "unable_to_verify_identity",
    69: "verify_my_identity",
    70: "verify_source_of_funds",
    71: "verify_top_up",
    72: "virtual_card_not_working",
    73: "visa_or_mastercard",
    74: "why_verify_identity",
    75: "wrong_amount_of_cash_received",
    76: "wrong_exchange_rate_for_cash_withdrawal"
}

In [ ]:
from transformers import BertTokenizer
import torch

# Load pre-trained BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [ ]:
import pandas as pd
from datasets import Dataset

df = train_data
df = pd.DataFrame(df)
df = df.rename(columns={'label': 'category_id_old'})
# df['label'] = df['label'].astype('category')

def clean_text(content):
    content = re.sub(r"https?://\S+", "", content)
    content = re.sub(r"what's", "what is", content)
    content = re.sub(r"Won't", "will not", content)
    content = re.sub(r"can't", "can not", content)
    content = re.sub(r"\'s", " ", content)
    content = re.sub(r"\'ve", " have", content)
    content = re.sub(r"n't", " not", content)
    content = re.sub(r"i'm", "i am", content)
    content = re.sub(r"\'re", " are", content)
    content = re.sub(r"\'d", " would", content)
    content = re.sub(r"\'ll", " will", content)
    content = re.sub(r"e - mail", "email", content)
    content = re.sub(r"\d+ ", "NUM", content)
    content = re.sub(r"<br />", '', content)
    content = re.sub(r'[\u0000-\u0019\u0021-\u0040\u007a-\uffff]', '', content)  # Remove special characters
    return content

def remove_punctuation(line):
    line = str(line)
    if line.strip()=='':
        return ''
    rule = re.compile("[^0-9a-zA-Z\s-]")
    line = rule.sub('',line)
    return line

df['category'] = df['category_id_old'].map(category_mapping_num_to_category)

df['clean_review'] = df['text'].apply(clean_text) #First data cleaning
print("Total missing values in the 'category' column: %d" % df['category'].isnull().sum())
print("Total missing values in the 'text' column: %d" % df['text'].isnull().sum())
df[df.isnull().values == True]
df = df[pd.notnull(df['text'])]
df['clean_review'] = df['clean_review'].apply(remove_punctuation) #Second data cleaning

value_counts = df['category'].value_counts()
sorted_values = value_counts.sort_values(ascending=False).index
mapping = {value: index for index, value in enumerate(sorted_values)}
df['category_id'] = df['category'].map(mapping)

df

Total missing values in the 'category' column: 0
Total missing values in the 'text' column: 0


,text,category_id_old,category,clean_review,category_id
0,I am still waiting on my card?,11,card_arrival,I am still waiting on my card,23
1,What can I do if my card still hasn't arrived ...,11,card_arrival,What can I do if my card still has not arrived...,23
2,I have been waiting over a week. Is the card s...,11,card_arrival,I have been waiting over a week Is the card st...,23
3,Can I track my card while it is in the process...,11,card_arrival,Can I track my card while it is in the process...,23
4,"How do I know if I will get my card, or if it ...",11,card_arrival,How do I know if I will get my card or if it i...,23
...,...,...,...,...,...
9998,You provide support in what countries?,24,country_support,You provide support in what countries,36
9999,What countries are you supporting?,24,country_support,What countries are you supporting,36
10000,What countries are getting support?,24,country_support,What countries are getting support,36
10001,Are cards available in the EU?,24,country_support,Are cards available in the EU,36


In [ ]:
d = {'category':df['category'].value_counts().index,'category_id_old':df['category_id_old'].value_counts().index, 'count': df['category'].value_counts()}
df_category = pd.DataFrame(data=d).reset_index(drop=True)
df_category

,category,category_id_old,count
0,card_payment_fee_charged,15,187
1,direct_debit_payment_not_recognised,28,182
2,balance_not_updated_after_cheque_or_cash_deposit,6,181
3,wrong_amount_of_cash_received,75,180
4,cash_withdrawal_charge,19,177
...,...,...,...
72,lost_or_stolen_card,41,82
73,card_swallowed,18,61
74,card_acceptance,10,59
75,virtual_card_not_working,72,41


In [ ]:
# Set the most frequently used 50,000 words (in `texts_to_matrix`, the top `MAX_NB_WORDS` words will be selected, and the first `MAX_NB_WORDS` columns will be taken)
MAX_NB_WORDS = 50000
# Maximum length of each `cut_review`
MAX_SEQUENCE_LENGTH = 250
# Set the dimension of the Embedding layer
EMBEDDING_DIM = 100

tokenizer = Tokenizer(num_words=MAX_NB_WORDS, filters='!"#$%&()*+,-./:;<=>?@[\]^_`{|}~', lower=True)
tokenizer.fit_on_texts(df['clean_review'].values)
word_index = tokenizer.word_index
print('Total of %s unique words.' % len(word_index))

Total of 2405 unique words.


In [ ]:
X = tokenizer.texts_to_sequences(df['clean_review'].values)
#Pad X to ensure uniform length across all columns
X = pad_sequences(X, maxlen=MAX_SEQUENCE_LENGTH)

#One-hot expansion of multi-class labels
Y = pd.get_dummies(df['category_id']).values

print(X.shape)
print(Y.shape)

(10003, 250)
(10003, 77)


**BERT model construction**

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer
from torch.utils.data import Dataset
import torch
import pandas as pd

print(len(df['clean_review']))
print(len(df['category_id']))
print(df['clean_review'].isnull().sum())
print(df['category_id'].isnull().sum())

df = df.dropna(subset=['clean_review', 'category_id'])

# Initialize the tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Define maximum length
max_length = 64

# Manually process text data
input_ids = []
attention_masks = []
processed_text_count = 0

for text in df['clean_review'].values:
    #Tokenization
    tokens = tokenizer.tokenize(text)

    # Add special tokens [CLS] and [SEP]
    tokens = ['[CLS]'] + tokens + ['[SEP]']

    # Truncate sequence
    if len(tokens) > max_length:
        tokens = tokens[:max_length]

    # Pad sequence
    padding_length = max_length - len(tokens)
    tokens = tokens + ['[PAD]'] * padding_length

    #Generate attention mask
    mask = [1 if token != '[PAD]' else 0 for token in tokens]

    # Convert tokens to IDs
    token_ids = tokenizer.convert_tokens_to_ids(tokens)

    input_ids.append(torch.tensor(token_ids))
    attention_masks.append(torch.tensor(mask))
    processed_text_count += 1

print(f"Processed text count: {processed_text_count}")

# Convert to tensors
input_ids = torch.stack(input_ids)
attention_masks = torch.stack(attention_masks)
Y = torch.tensor(df['category_id'].values)

# Check data length
print(len(input_ids))
print(len(attention_masks))
print(len(Y))

# Ensure uniform length
assert len(input_ids) == len(attention_masks) == len(Y), "Data lengths do not match!"

# Use all data for training
X_train = input_ids
Y_train = Y
attention_masks_train = attention_masks

# Define a custom dataset
class TextDataset(Dataset):
    def __init__(self, input_ids, attention_masks, labels):
        self.input_ids = input_ids
        self.attention_masks = attention_masks
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_masks[idx],
            'labels': self.labels[idx]
        }

# Create the dataset
train_dataset = TextDataset(X_train, attention_masks_train, Y_train)

# Set up BERT model
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=df['category_id'].nunique())

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Enable mixed precision (fp16) training
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    report_to="none",
    fp16=True  # Enable mixed precision training
)

# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

# Train the model
trainer.train()

# Save the trained model
model.save_pretrained('./saved_model')

10003
10003
0
0
Processed text count: 10003
10003
10003
10003


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
10,4.405700
20,4.353300
30,4.342500
40,4.362500
50,4.368800
60,4.397300
70,4.362000
80,4.386900
90,4.355000
100,4.321900


Step,Training Loss
10,4.405700
20,4.353300
30,4.342500
40,4.362500
50,4.368800
60,4.397300
70,4.362000
80,4.386900
90,4.355000
100,4.321900


In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset

# Load the tokenizer and the trained model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Convert model to float16 and move to GPU
model = BertForSequenceClassification.from_pretrained('./saved_model').to(device).half()
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print(mapping)
# Load the test dataset
# test_df = pd.read_csv('./test.csv')
test_df = test_data
test_df = pd.DataFrame(test_df)
test_df = test_df.rename(columns={'label': 'category_id_old'})
test_df['category'] = test_df['category_id_old'].map(category_mapping_num_to_category)
test_df['category_id'] = test_df['category'].map(mapping)
# Ensure the correct column name is used
if 'text' not in test_df.columns:
    raise ValueError(f"Expected 'text' column, but found: {test_df.columns.tolist()}")



test_df



{'card_payment_fee_charged': 0, 'direct_debit_payment_not_recognised': 1, 'balance_not_updated_after_cheque_or_cash_deposit': 2, 'wrong_amount_of_cash_received': 3, 'cash_withdrawal_charge': 4, 'transaction_charged_twice': 5, 'declined_cash_withdrawal': 6, 'transfer_fee_charged': 7, 'balance_not_updated_after_bank_transfer': 8, 'transfer_not_received_by_recipient': 9, 'request_refund': 10, 'card_payment_not_recognised': 11, 'card_payment_wrong_exchange_rate': 12, 'extra_charge_on_statement': 13, 'wrong_exchange_rate_for_cash_withdrawal': 14, 'Refund_not_showing_up': 15, 'reverted_card_payment?': 16, 'cash_withdrawal_not_recognised': 17, 'activate_my_card': 18, 'pending_card_payment': 19, 'cancel_transfer': 20, 'beneficiary_not_allowed': 21, 'declined_card_payment': 22, 'card_arrival': 23, 'pending_top_up': 24, 'pending_transfer': 25, 'top_up_reverted': 26, 'top_up_failed': 27, 'pending_cash_withdrawal': 28, 'card_linking': 29, 'failed_transfer': 30, 'visa_or_mastercard': 31, 'declined_

,text,category_id_old,category,category_id
0,How do I locate my card?,11,card_arrival,23
1,"I still have not received my new card, I order...",11,card_arrival,23
2,I ordered a card but it has not arrived. Help ...,11,card_arrival,23
3,Is there a way to know when my card will arrive?,11,card_arrival,23
4,My card has not arrived yet.,11,card_arrival,23
...,...,...,...,...
3075,"If i'm not in the UK, can I still get a card?",24,country_support,36
3076,How many countries do you support?,24,country_support,36
3077,What countries do you do business in?,24,country_support,36
3078,What are the countries you operate in.,24,country_support,36


In [ ]:
# Tokenize all test data at once
encoded_inputs = tokenizer(test_df['text'].tolist(), padding="max_length", truncation=True, max_length=64, return_tensors="pt")

# Create DataLoader for batch processing
test_dataset = TensorDataset(encoded_inputs["input_ids"], encoded_inputs["attention_mask"])
test_loader = DataLoader(test_dataset, batch_size=32)  # Batch processing for faster inference

# Perform inference
predictions = []
model.eval()  # Set model to evaluation mode

with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_masks = batch
        input_ids, attention_masks = input_ids.to(device), attention_masks.to(device)  # Move data to GPU

        outputs = model(input_ids, attention_mask=attention_masks)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        predictions.extend(preds)

test_df['predicted_label'] = [p for p in predictions]
test_df.to_csv("test_predictions.csv", index=False)

print("Predictions saved to test_predictions.csv")

Predictions saved to test_predictions.csv


In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.metrics import classification_report, accuracy_score
from torch.utils.data import Dataset, DataLoader
import pandas as pd

# Load the tokenizer and the trained model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Convert model to float16 and move to GPU
model = BertForSequenceClassification.from_pretrained('./saved_model').to(device).half()
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Load the test dataset
test_df = test_data
test_df = pd.DataFrame(test_df)
test_df = test_df.rename(columns={'label': 'category_id_old'})
test_df['category'] = test_df['category_id_old'].map(category_mapping_num_to_category)
test_df['category_id'] = test_df['category'].map(mapping)

# Extract labels as tensor
Y_test = torch.tensor(test_df['category_id'].values)
category_names = test_df['category_id'].unique().tolist()  # List of category names
category_names = [str(name) for name in category_names]
print(category_names)
# Tokenize the test data
encoded_inputs = tokenizer(test_df['text'].tolist(), padding="max_length", truncation=True, max_length=64, return_tensors="pt")

# Convert lists to tensors
input_ids = encoded_inputs["input_ids"].to(device)
attention_masks = encoded_inputs["attention_mask"].to(device)

# Define a custom dataset for testing
class TestDataset(Dataset):
    def __init__(self, input_ids, attention_masks):
        self.input_ids = input_ids
        self.attention_masks = attention_masks

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_masks[idx]
        }

# Create test dataset and DataLoader
test_dataset = TestDataset(input_ids, attention_masks)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Evaluation function
def evaluate(model, test_loader, Y_test, category_names):
    predictions = []

    # Set model to evaluation mode
    model.eval()

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            # Forward pass
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1).cpu().numpy()

            predictions.extend(preds)

    # Compute accuracy
    accuracy = accuracy_score(Y_test, predictions)

    # Print classification report
    report = classification_report(Y_test, predictions, target_names=category_names, digits=4)

    print(f"\nAccuracy: {accuracy:.6f}\n")
    print(report)

# Run evaluation and print results
evaluate(model, test_loader, Y_test, category_names)



['23', '29', '56', '12', '13', '28', '40', '55', '38', '57', '49', '72', '59', '50', '76', '58', '24', '20', '67', '3', '0', '9', '35', '66', '74', '26', '2', '11', '47', '44', '65', '61', '31', '64', '43', '71', '70', '1', '62', '6', '19', '46', '10', '32', '15', '22', '25', '60', '73', '5', '54', '37', '16', '42', '21', '7', '69', '30', '53', '41', '34', '52', '48', '75', '14', '68', '27', '8', '17', '45', '51', '18', '4', '33', '39', '63', '36']

Accuracy: 0.935390

              precision    recall  f1-score   support

          23     0.8636    0.9500    0.9048        40
          29     0.8293    0.8500    0.8395        40
          56     0.9737    0.9250    0.9487        40
          12     1.0000    0.9000    0.9474        40
          13     0.9500    0.9500    0.9500        40
          28     0.9091    1.0000    0.9524        40
          40     0.8333    1.0000    0.9091        40
          55     0.9268    0.9500    0.9383        40
          38     0.8095    0.8500    0.

In [ ]:
print(mapping)
print('-'*100)
print(category_mapping_num_to_category)

{'card_payment_fee_charged': 0, 'direct_debit_payment_not_recognised': 1, 'balance_not_updated_after_cheque_or_cash_deposit': 2, 'wrong_amount_of_cash_received': 3, 'cash_withdrawal_charge': 4, 'transaction_charged_twice': 5, 'declined_cash_withdrawal': 6, 'transfer_fee_charged': 7, 'balance_not_updated_after_bank_transfer': 8, 'transfer_not_received_by_recipient': 9, 'request_refund': 10, 'card_payment_not_recognised': 11, 'card_payment_wrong_exchange_rate': 12, 'extra_charge_on_statement': 13, 'wrong_exchange_rate_for_cash_withdrawal': 14, 'Refund_not_showing_up': 15, 'reverted_card_payment?': 16, 'cash_withdrawal_not_recognised': 17, 'activate_my_card': 18, 'pending_card_payment': 19, 'cancel_transfer': 20, 'beneficiary_not_allowed': 21, 'declined_card_payment': 22, 'card_arrival': 23, 'pending_top_up': 24, 'pending_transfer': 25, 'top_up_reverted': 26, 'top_up_failed': 27, 'pending_cash_withdrawal': 28, 'card_linking': 29, 'failed_transfer': 30, 'visa_or_mastercard': 31, 'declined_

In [ ]:
reversed_mapping = {value: key for key, value in mapping.items()}
reversed_category_mapping_num_to_category = {value2: key2 for key2, value2 in category_mapping_num_to_category.items()}
print(reversed_mapping)
print(reversed_category_mapping_num_to_category)

{0: 'card_payment_fee_charged', 1: 'direct_debit_payment_not_recognised', 2: 'balance_not_updated_after_cheque_or_cash_deposit', 3: 'wrong_amount_of_cash_received', 4: 'cash_withdrawal_charge', 5: 'transaction_charged_twice', 6: 'declined_cash_withdrawal', 7: 'transfer_fee_charged', 8: 'balance_not_updated_after_bank_transfer', 9: 'transfer_not_received_by_recipient', 10: 'request_refund', 11: 'card_payment_not_recognised', 12: 'card_payment_wrong_exchange_rate', 13: 'extra_charge_on_statement', 14: 'wrong_exchange_rate_for_cash_withdrawal', 15: 'Refund_not_showing_up', 16: 'reverted_card_payment?', 17: 'cash_withdrawal_not_recognised', 18: 'activate_my_card', 19: 'pending_card_payment', 20: 'cancel_transfer', 21: 'beneficiary_not_allowed', 22: 'declined_card_payment', 23: 'card_arrival', 24: 'pending_top_up', 25: 'pending_transfer', 26: 'top_up_reverted', 27: 'top_up_failed', 28: 'pending_cash_withdrawal', 29: 'card_linking', 30: 'failed_transfer', 31: 'visa_or_mastercard', 32: 'decli

In [ ]:
def predict_sentences(sentences, model, tokenizer, device, max_length=64, batch_size=32):
    # Tokenize all input sentences at once
    encoded_inputs = tokenizer(sentences, padding="max_length", truncation=True, max_length=max_length, return_tensors="pt")

    # Create DataLoader for batch processing
    test_dataset = TensorDataset(encoded_inputs["input_ids"], encoded_inputs["attention_mask"])
    test_loader = DataLoader(test_dataset, batch_size=batch_size)  # Batch processing for faster inference

    # Perform inference
    predictions = []
    model.eval()  # Set model to evaluation mode

    with torch.no_grad():
        for batch in test_loader:
            input_ids, attention_masks = batch
            input_ids, attention_masks = input_ids.to(device), attention_masks.to(device)  # Move data to GPU

            outputs = model(input_ids, attention_mask=attention_masks)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            predictions.extend(preds)

    # Convert predictions to labels if category mapping is provided
    predicted_labels = [p for p in predictions]

    return predicted_labels

sentences = ["'my card has not arrived yet"]

# Call the prediction function
predicted_category_id = predict_sentences(sentences, model, tokenizer, device)
translated_category = [reversed_mapping.get(num) for num in predicted_category_id if num in reversed_mapping]
translated_category_id_old = [reversed_category_mapping_num_to_category.get(category_en) for category_en in translated_category if category_en in reversed_category_mapping_num_to_category]
print()
print("Predicted category_id:", predicted_category_id)
print("Predicted category_id_old:", translated_category_id_old)
print("Predicted category:", translated_category)


Predicted category_id: [23]
Predicted category_id_old: [11]
Predicted category: ['card_arrival']
